# Notebook 13 — Feature Engineering Pipeline
Everything up to this point was demonstrated step-by-step for learning clarity. In
real production systems, that's fragile — it's easy to accidentally fit a transformer
on the wrong split. This notebook builds a **single reusable pipeline** that a senior
engineer would actually ship: leakage-safe by construction, and reproducible on any
new batch of customers.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
customers["churn_binary"] = (customers["churn"]=="Yes").astype(int)
customers.shape

## 1. Train/Test Separation — Always First

This is non-negotiable, and it happens **before** any transformer (imputer, scaler,
encoder) is fit. Everything downstream in this notebook respects that split.

In [ ]:
target = "churn_binary"
drop_cols = ["customer_id","signup_date","churn","support_ticket_text",target]
feature_cols = [c for c in customers.columns if c not in drop_cols]

X = customers[feature_cols]
y = customers[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 2. Custom Feature-Creation Step (a Transformer)

We wrap our engineered numeric/date features (from Notebooks 2, 4, 6) inside a
`FunctionTransformer` so they're computed identically and automatically on any future
batch of data — training, validation, or live production — using only information
available in that row plus fixed reference constants.

In [ ]:
REFERENCE_DATE = pd.Timestamp("2024-06-30")  # fixed, not recomputed at call time

def engineer_features(df):
    df = df.copy()
    df["monthly_charges"] = df["monthly_charges"].fillna(df["monthly_charges"].median())
    df["charge_per_tenure_month"] = df["monthly_charges"] / df["tenure_months"].replace(0, 1)
    df["contract_ordinal"] = df["contract"].map({"Month-to-month":0,"One year":1,"Two year":2})
    df["is_electronic_check"] = (df["payment_method"]=="Electronic check").astype(int)
    df["is_fiber"] = (df["internet_service"]=="Fiber optic").astype(int)
    df["log_total_charges"] = np.log1p(df["total_charges"].fillna(0))
    return df

feature_creator = FunctionTransformer(engineer_features)

## 3. Numerical & Categorical Transformation Branches (ColumnTransformer)

`ColumnTransformer` applies **different** preprocessing to different column types in
one coherent step, and — critically — every sub-transformer's `.fit()` will only ever
see training data when used inside a `Pipeline.fit()` call.

In [ ]:
numeric_features = ["monthly_charges","tenure_months","total_charges",
                     "charge_per_tenure_month","log_total_charges","senior_citizen"]
categorical_features = ["gender","partner","dependents","internet_service",
                         "multiple_lines","phone_service"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),  # production-safe for unseen categories
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

## 4. Full Pipeline — Feature Creation → Preprocessing → Feature Selection → Model

In [ ]:
full_pipeline = Pipeline([
    ("feature_creation", feature_creator),
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=15)),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)),
])

full_pipeline.fit(X_train, y_train)
preds = full_pipeline.predict(X_test)
probs = full_pipeline.predict_proba(X_test)[:,1]

print("Test ROC-AUC:", round(roc_auc_score(y_test, probs), 4))
print(classification_report(y_test, preds))

## 5. Why This Structure Prevents Leakage

- `feature_creation` uses only per-row arithmetic and a **fixed** reference date — no
  dependency on the full dataset's statistics.
- `SimpleImputer` and `StandardScaler` are fit **inside** `full_pipeline.fit(X_train,
  y_train)` — they never see `X_test` during fitting, only during `.transform()`.
- `SelectKBest` is likewise fit only on the training fold's mutual information with
  `y_train`.
- Calling `full_pipeline.predict(X_test)` reuses the *already-fitted* transformers —
  it does not refit anything on test data.

This is the exact same pipeline object that would be pickled and deployed to serve
predictions on brand-new customers in production, guaranteeing training/serving
consistency.

## 6. Reusability — Applying the Pipeline to "New" Data

In [ ]:
# Simulate 5 brand-new customer rows arriving in production
new_customers = X_test.sample(5, random_state=1).reset_index(drop=True)
new_predictions = full_pipeline.predict_proba(new_customers)[:,1]

pd.DataFrame({"predicted_churn_probability": new_predictions.round(3)})

## Summary

This pipeline is the deliverable a senior engineer would actually hand off to an MLOps
team: one object, one `.fit()` call, one `.predict()` call, leakage-safe by
construction, and reusable on any future batch without re-deriving preprocessing
logic by hand.